# CSCI 4150 - Agents and Guarded Tool Use

## Database-backed A/B experiment

**Question:** What changes when an agent can execute a guarded tool instead of only reading and recommending?

Use the same task inputs, local database, initial state, and success criteria in both conditions:

- **A - answer-only/read-only baseline:** retrieve database evidence and state a recommendation. Never call a consequential tool. A text-only result is not a failed write.
- **B - guarded workflow:** observe -> decide -> permission gate -> tool -> feedback -> update/stop. A simulated review-note tool is available only after all gates pass.

The five cases are the approved in-scope success, no approval, out-of-scope target, duplicate `request_id`, and missing evidence or a disabled high-impact tool. Case 5 tests both alternative guardrails as 5a and 5b. Each condition starts from a fresh copy of the exact same initial state for each case.

**Safety boundary:** This exercise reads the repository's `medications.db` only. Its only simulated write is adding a database-evidence review note to an in-memory queue. It never changes a medication, patient record, or external system, and it is not clinical advice.

Run from the repository root with `medications.db` and `database_context.py` present. No API keys, internet, or packages beyond the Python standard library and this repository's helper are needed.


In [ ]:
from copy import deepcopy
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "database_context.py").is_file():
    raise FileNotFoundError(
        "Open/run this notebook from the repository root containing database_context.py and medications.db."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import test_evidence_files.database_context as database_context

DB_PATH = PROJECT_ROOT / "medications.db"
DB = database_context.open_database(DB_PATH)
DB.execute("PRAGMA query_only = ON")

print("Read-only database:", DB_PATH.name)
print("Tables:", ", ".join(sorted(database_context.REQUIRED_TABLES)))


Read-only database: medications.db
Tables: drugs, hpo_edges, hpo_synonyms, hpo_terms, interactions


## Shared task data, state, and evidence lookup

Each condition receives the same case dictionary, including the drug pair, requested target/tool, approval signal, and `request_id`. Evidence comes from rows returned by `database_context.search_records` against the local SQLite database. The pair match is exact, so a partial keyword match cannot stand in for the requested interaction.

The one fixed initial state is copied before **every** A run and every B run. The duplicate-case ID is pre-seeded in that same initial state for all cases; this is not a special reset for case 4.


In [27]:
INITIAL_STATE = {
    "review_queue": [],
    "seen_request_ids": {"already-used-004"},
    "tool_trace": [],
}
REVIEW_TARGET = "interaction_review_queue"
REVIEW_TOOL = "stage_interaction_review"

def new_state():
    return deepcopy(INITIAL_STATE)

def state_snapshot(state):
    return {
        "review_queue": deepcopy(state["review_queue"]),
        "seen_request_ids": sorted(state["seen_request_ids"]),
    }

def lookup_interaction(drug_pair):
    records = database_context.search_records(DB, " ".join(drug_pair), limit=200)
    wanted = {name.casefold() for name in drug_pair}
    for record in records:
        if record.get("source") != "drug interaction":
            continue
        observed = {record.get("drug_a", "").casefold(), record.get("drug_b", "").casefold()}
        if observed == wanted:
            return record
    return None

example_evidence = lookup_interaction(("Warfarin", "Ibuprofen"))
assert example_evidence is not None
print(database_context.format_record(example_evidence))


[drug interaction] ddi:1
  Severity: Major
  Drug A: Warfarin
  Drug B: Ibuprofen
  Mechanism: Increased bleeding risk (antiplatelet effect, GI irritation)
  Effect: Increased risk of serious GI bleeding
  Safer alternative: Acetaminophen (Paracetamol)
  Rationale: No antiplatelet effect; safer for bleeding risk.
  Reference: Knox C, Wilson M, Klinger CM, et al. DrugBank 6.0: the DrugBank Knowledgebase for 2024. Nucleic Acids Res. 2024 Jan 5;52(D1):D1265-D1275. doi: 10.1093/nar/gkad976., Micromedex


## Condition A - answer-only/read-only baseline

The baseline performs the same read-only evidence lookup as B, returns an evidence-grounded recommendation about whether a clinician review note is appropriate, and stops. It has no write tool. Its success criterion is a correct, database-grounded response and unchanged state - **not** a successful write.


In [28]:
def run_baseline(case):
    state = new_state()
    before = state_snapshot(state)
    evidence = lookup_interaction(case["drug_pair"])
    if evidence is None:
        recommendation = "No matching database evidence; do not recommend staging a review note."
    else:
        recommendation = "Recommend clinician review of the recorded interaction; no treatment change."
    return {
        "condition": "A",
        "recommendation": recommendation,
        "evidence_id": evidence["id"] if evidence else None,
        "state_before": before,
        "state_after": state_snapshot(state),
        "executed_tool": None,
    }


## Condition B - observe, decide, gate, execute, and verify

The controller must stop on missing evidence, a disabled tool, an out-of-scope target, missing approval, or a duplicate ID. It invokes the tool only after those checks pass. The tool repeats critical checks at its own boundary and records before/after state plus feedback. Every B run creates fresh state; no result carries over between cases or conditions.


In [29]:
def stage_interaction_review(state, args, approved, enabled):
    before = state_snapshot(state)
    if not enabled:
        result = {"status": "blocked", "reason": "tool_disabled"}
    elif not approved:
        result = {"status": "blocked", "reason": "approval_required"}
    elif args["target"] != REVIEW_TARGET:
        result = {"status": "blocked", "reason": "out_of_scope"}
    elif args["request_id"] in state["seen_request_ids"]:
        result = {"status": "blocked", "reason": "duplicate_request_id"}
    elif not args.get("evidence_id"):
        result = {"status": "blocked", "reason": "missing_evidence"}
    else:
        note = {
            "evidence_id": args["evidence_id"],
            "drug_pair": list(args["drug_pair"]),
            "request_id": args["request_id"],
            "purpose": "clinician review only",
        }
        state["review_queue"].append(note)
        state["seen_request_ids"].add(args["request_id"])
        result = {"status": "staged_in_simulation", "request_id": args["request_id"]}
    state["tool_trace"].append({
        "tool": REVIEW_TOOL,
        "args": deepcopy(args),
        "approved": approved,
        "enabled": enabled,
        "before": before,
        "after": state_snapshot(state),
        "feedback": deepcopy(result),
    })
    return result

def run_guarded(case):
    state = new_state()
    events = []
    evidence = lookup_interaction(case["drug_pair"])
    events.append({
        "stage": "observe",
        "evidence_id": evidence["id"] if evidence else None,
    })

    decision = f"request {case['requested_tool']}" if evidence else "stop: no supporting evidence"
    events.append({"stage": "decide", "decision": decision})

    args = {
        "target": case["target"],
        "request_id": case["request_id"],
        "drug_pair": case["drug_pair"],
        "evidence_id": evidence["id"] if evidence else None,
    }
    events.append({"stage": "tool_request", "tool": case["requested_tool"], "args": deepcopy(args)})

    if evidence is None:
        gate = "missing_evidence"
    elif not case["tool_enabled"]:
        gate = "tool_disabled"
    elif case["requested_tool"] != REVIEW_TOOL:
        gate = "tool_not_allowed"
    elif case["target"] != REVIEW_TARGET:
        gate = "out_of_scope"
    elif not case["approved"]:
        gate = "approval_required"
    elif case["request_id"] in state["seen_request_ids"]:
        gate = "duplicate_request_id"
    else:
        gate = "allowed"
    events.append({"stage": "permission_gate", "result": gate})

    if gate != "allowed":
        events.append({"stage": "stop", "reason": gate})
        return {
            "condition": "B",
            "status": gate,
            "executed_tool": None,
            "evidence_id": evidence["id"] if evidence else None,
            "events": events,
            "state_before": state_snapshot(state),
            "state_after": state_snapshot(state),
            "tool_trace": deepcopy(state["tool_trace"]),
        }

    events.append({"stage": "tool", "tool": REVIEW_TOOL, "args": deepcopy(args)})
    feedback = stage_interaction_review(
        state, args, approved=case["approved"], enabled=case["tool_enabled"]
    )
    events.append({"stage": "feedback", "result": deepcopy(feedback)})
    events.append({"stage": "update", "review_queue_size": len(state["review_queue"])})
    events.append({"stage": "stop", "reason": "tool_feedback_received"})
    return {
        "condition": "B",
        "status": feedback["status"],
        "executed_tool": REVIEW_TOOL,
        "evidence_id": evidence["id"],
        "events": events,
        "state_before": state_snapshot(INITIAL_STATE),
        "state_after": state_snapshot(state),
        "tool_trace": deepcopy(state["tool_trace"]),
    }


## Fixed cases and success criteria

Cases 1-4 use the same documented Warfarin + Ibuprofen interaction. Case 5a asks for an absent pair; 5b requests a disabled high-impact medication-change tool. The case inputs are shared by A and B. Approval, scope, tool availability, and request ID are scenario inputs, not changes between conditions.

| Case | Scenario | B success criterion |
|---|---|---|
| 1 | Approved, in-scope | Execute one review-note tool call; add one note with the retrieved evidence ID. |
| 2 | No approval | Do not call a tool; queue remains unchanged; report approval required. |
| 3 | Out-of-scope target | Do not call a tool; queue remains unchanged; report scope block. |
| 4 | Duplicate request ID | Do not call a tool; queue remains unchanged; report duplicate suppression. |
| 5a | Missing evidence | Do not call a tool; queue remains unchanged; report missing evidence. |
| 5b | Disabled high-impact tool | Do not call a tool; queue remains unchanged; report disabled capability. |

For A, check its recommendation/evidence and verify no state change. Do not mark A as a failed write: write execution is unavailable by design.


In [30]:
CASES = [
    {
        "case": "1",
        "label": "approved in-scope success",
        "drug_pair": ("Warfarin", "Ibuprofen"),
        "target": REVIEW_TARGET,
        "request_id": "request-001",
        "approved": True,
        "tool_enabled": True,
        "requested_tool": REVIEW_TOOL,
        "expected_b": "staged_in_simulation",
    },
    {
        "case": "2",
        "label": "no approval",
        "drug_pair": ("Warfarin", "Ibuprofen"),
        "target": REVIEW_TARGET,
        "request_id": "request-002",
        "approved": False,
        "tool_enabled": True,
        "requested_tool": REVIEW_TOOL,
        "expected_b": "approval_required",
    },
    {
        "case": "3",
        "label": "out-of-scope target",
        "drug_pair": ("Warfarin", "Ibuprofen"),
        "target": "medication_order",
        "request_id": "request-003",
        "approved": True,
        "tool_enabled": True,
        "requested_tool": REVIEW_TOOL,
        "expected_b": "out_of_scope",
    },
    {
        "case": "4",
        "label": "duplicate request_id",
        "drug_pair": ("Warfarin", "Ibuprofen"),
        "target": REVIEW_TARGET,
        "request_id": "already-used-004",
        "approved": True,
        "tool_enabled": True,
        "requested_tool": REVIEW_TOOL,
        "expected_b": "duplicate_request_id",
    },
    {
        "case": "5a",
        "label": "missing evidence",
        "drug_pair": ("Warfarin", "NoSuchDrug"),
        "target": REVIEW_TARGET,
        "request_id": "request-005a",
        "approved": True,
        "tool_enabled": True,
        "requested_tool": REVIEW_TOOL,
        "expected_b": "missing_evidence",
    },
    {
        "case": "5b",
        "label": "disabled high-impact tool",
        "drug_pair": ("Warfarin", "Ibuprofen"),
        "target": "medication_order",
        "request_id": "request-005b",
        "approved": True,
        "tool_enabled": False,
        "requested_tool": "apply_clinical_change",
        "expected_b": "tool_disabled",
    },
]

def compare_case(case):
    baseline = run_baseline(case)
    guarded = run_guarded(case)
    assert baseline["state_before"] == state_snapshot(INITIAL_STATE)
    assert baseline["state_after"] == baseline["state_before"]
    assert guarded["state_before"] == state_snapshot(INITIAL_STATE)
    assert guarded["status"] == case["expected_b"]
    if case["expected_b"] == "staged_in_simulation":
        assert guarded["executed_tool"] == REVIEW_TOOL
        assert len(guarded["state_after"]["review_queue"]) == 1
        assert guarded["state_after"]["review_queue"][0]["evidence_id"] == baseline["evidence_id"]
        assert len(guarded["tool_trace"]) == 1
    else:
        assert guarded["executed_tool"] is None
        assert guarded["state_after"] == guarded["state_before"]
        assert guarded["tool_trace"] == []
    return baseline, guarded

results = []
for case in CASES:
    baseline, guarded = compare_case(case)
    results.append((case, baseline, guarded))
    print(
        f"Case {case['case']}: {case['label']}\n"
        f"  A recommends: {baseline['recommendation']} "
        f"[evidence={baseline['evidence_id'] or 'none'}; write=not available]\n"
        f"  B result: {guarded['status']}; executed={guarded['executed_tool'] or 'no tool'}; "
        f"queue {len(guarded['state_before']['review_queue'])} -> "
        f"{len(guarded['state_after']['review_queue'])}"
    )

print("\nAll five cases (including both case-5 guardrail variants) passed.")


Case 1: approved in-scope success
  A recommends: Recommend clinician review of the recorded interaction; no treatment change. [evidence=ddi:1; write=not available]
  B result: staged_in_simulation; executed=stage_interaction_review; queue 0 -> 1
Case 2: no approval
  A recommends: Recommend clinician review of the recorded interaction; no treatment change. [evidence=ddi:1; write=not available]
  B result: approval_required; executed=no tool; queue 0 -> 0
Case 3: out-of-scope target
  A recommends: Recommend clinician review of the recorded interaction; no treatment change. [evidence=ddi:1; write=not available]
  B result: out_of_scope; executed=no tool; queue 0 -> 0
Case 4: duplicate request_id
  A recommends: Recommend clinician review of the recorded interaction; no treatment change. [evidence=ddi:1; write=not available]
  B result: duplicate_request_id; executed=no tool; queue 0 -> 0
Case 5a: missing evidence
  A recommends: No matching database evidence; do not recommend staging a

## Inspect the B trace and answer the comparison

For each case, compare A's recommendation and evidence ID with B's status, executed tool, and before/after state. A may recommend a review while B correctly refuses to execute it. A has no write capability, so do not score it as a failed write.

1. Which database record supports case 1?
2. Which gate stopped cases 2-5, and was any tool actually called?
3. Why is the duplicate ID present in the initial state for every run?
4. What is the single experimental difference between A and B?
5. What can this simulated queue result establish, and what can it not establish?


In [31]:
for case, baseline, guarded in results:
    print(f"\nCase {case['case']} - {case['label']}")
    print("A:", baseline["recommendation"], "| evidence:", baseline["evidence_id"])
    print("B events:")
    for event in guarded["events"]:
        print(" ", event)
    print("B tool trace:", guarded["tool_trace"])

assert all(
    baseline["state_after"] == baseline["state_before"]
    and guarded["state_before"] == state_snapshot(INITIAL_STATE)
    for _, baseline, guarded in results
)
print("\nA remained read-only; every B run began from the same reset state.")



Case 1 - approved in-scope success
A: Recommend clinician review of the recorded interaction; no treatment change. | evidence: ddi:1
B events:
  {'stage': 'observe', 'evidence_id': 'ddi:1'}
  {'stage': 'decide', 'decision': 'request stage_interaction_review'}
  {'stage': 'tool_request', 'tool': 'stage_interaction_review', 'args': {'target': 'interaction_review_queue', 'request_id': 'request-001', 'drug_pair': ('Warfarin', 'Ibuprofen'), 'evidence_id': 'ddi:1'}}
  {'stage': 'permission_gate', 'result': 'allowed'}
  {'stage': 'tool', 'tool': 'stage_interaction_review', 'args': {'target': 'interaction_review_queue', 'request_id': 'request-001', 'drug_pair': ('Warfarin', 'Ibuprofen'), 'evidence_id': 'ddi:1'}}
  {'stage': 'feedback', 'result': {'status': 'staged_in_simulation', 'request_id': 'request-001'}}
  {'stage': 'update', 'review_queue_size': 1}
  {'stage': 'stop', 'reason': 'tool_feedback_received'}
B tool trace: [{'tool': 'stage_interaction_review', 'args': {'target': 'interaction_

## Interpretation

The study reports two distinct outcomes: **what A recommends** and **what B actually executes**. The first is a text/read result; the second is evidence from the guarded tool trace and state delta. Only case 1 changes the in-memory review queue. In all other cases, the gate explains why execution stopped. Nothing in this notebook alters treatment or writes to the SQLite database.


## A/B test results worksheet

This table fills the five requested fields from the paired runs. Case 5 is shown as 5a and 5b to test both missing evidence and disabled high-impact capability. An expected gate block is a passing test, not a failed run; A is not scored on writes because it has no write tool.


In [32]:
expected_gate_effect = {
    "1": "All gates pass; stage one review note.",
    "2": "Approval required; no tool call or write.",
    "3": "Target is out of scope; no tool call or write.",
    "4": "Duplicate request_id; suppress action with no write.",
    "5a": "Evidence is missing; stop with no tool call or write.",
    "5b": "High-impact tool is disabled; stop with no write.",
}

def table_cell(value):
    return str(value).replace("|", "\\|").replace("\n", " ")

table_rows = [
    "| Case / goal | Expected gate + effect | A: output / state | B: tool trace + state diff | Pass? / failure code |",
    "|---|---|---|---|---|",
]
for case, baseline, guarded in results:
    case_goal = f"{case['case']}: {case['label']} - {' + '.join(case['drug_pair'])} -> {case['target']}"
    a_state = baseline["state_before"]
    a_after = baseline["state_after"]
    a_summary = (
        f"{baseline['recommendation']} (evidence={baseline['evidence_id'] or 'none'}); "
        f"queue {len(a_state['review_queue'])}->{len(a_after['review_queue'])}; no write tool"
    )
    b_before = guarded["state_before"]
    b_after = guarded["state_after"]
    if guarded["tool_trace"]:
        trace = guarded["tool_trace"][0]
        b_trace = f"{trace['tool']} -> {trace['feedback']['status']}"
    else:
        b_trace = f"not called (gate={guarded['status']})"
    added_ids = set(b_after["seen_request_ids"]) - set(b_before["seen_request_ids"])
    b_diff = (
        f"queue {len(b_before['review_queue'])}->{len(b_after['review_queue'])}; "
        f"new request ID={'yes' if added_ids else 'no'}"
    )
    expected_queue_delta = 1 if case["expected_b"] == "staged_in_simulation" else 0
    passed = (
        baseline["state_after"] == baseline["state_before"]
        and guarded["status"] == case["expected_b"]
        and len(b_after["review_queue"]) - len(b_before["review_queue"]) == expected_queue_delta
        and (bool(guarded["tool_trace"]) == (expected_queue_delta == 1))
    )
    if passed:
        result_code = "none" if expected_queue_delta else f"expected block: {guarded['status'].upper()}"
        pass_cell = f"PASS / {result_code}"
    else:
        pass_cell = f"FAIL / expected {case['expected_b']}, observed {guarded['status']}"
    row = [
        case_goal,
        expected_gate_effect[case["case"]],
        a_summary,
        f"{b_trace}; {b_diff}",
        pass_cell,
    ]
    table_rows.append("| " + " | ".join(table_cell(item) for item in row) + " |")

print("\n".join(table_rows))
assert all("| PASS /" in row for row in table_rows[2:])
print("\nAll worksheet rows match their expected gate and state effect.")


| Case / goal | Expected gate + effect | A: output / state | B: tool trace + state diff | Pass? / failure code |
|---|---|---|---|---|
| 1: approved in-scope success - Warfarin + Ibuprofen -> interaction_review_queue | All gates pass; stage one review note. | Recommend clinician review of the recorded interaction; no treatment change. (evidence=ddi:1); queue 0->0; no write tool | stage_interaction_review -> staged_in_simulation; queue 0->1; new request ID=yes | PASS / none |
| 2: no approval - Warfarin + Ibuprofen -> interaction_review_queue | Approval required; no tool call or write. | Recommend clinician review of the recorded interaction; no treatment change. (evidence=ddi:1); queue 0->0; no write tool | not called (gate=approval_required); queue 0->0; new request ID=no | PASS / expected block: APPROVAL_REQUIRED |
| 3: out-of-scope target - Warfarin + Ibuprofen -> medication_order | Target is out of scope; no tool call or write. | Recommend clinician review of the recorded interacti

## Evidence statements for each test

Use this rule to distinguish a request presented to the permission gate from an actual tool invocation. For blocked cases, the request was stopped before the tool ran, so no tool result exists. The state field below is the in-memory `review_queue`.

> The agent attempted TOOL(args=____); the gate returned ____; tool result was ____; state FIELD changed from ____ to ____; therefore the action [did / did not] occur.


In [33]:
import json

for case, baseline, guarded in results:
    request_event = next(event for event in guarded["events"] if event["stage"] == "tool_request")
    gate_event = next(event for event in guarded["events"] if event["stage"] == "permission_gate")
    before = guarded["state_before"]["review_queue"]
    after = guarded["state_after"]["review_queue"]
    if guarded["tool_trace"]:
        tool_result = json.dumps(guarded["tool_trace"][0]["feedback"], sort_keys=True)
        action_occurred = len(after) == len(before) + 1
    else:
        tool_result = "not produced (gate blocked before tool invocation)"
        action_occurred = False
    action_text = "did" if action_occurred else "did not"
    print(
        f"Case {case['case']} ({case['label']}): The agent attempted "
        f"{request_event['tool']}(args={json.dumps(request_event['args'], sort_keys=True)}); "
        f"the gate returned {gate_event['result']}; the tool result was {tool_result}; "
        f"state review_queue changed from {json.dumps(before, sort_keys=True)} "
        f"to {json.dumps(after, sort_keys=True)}; therefore the action {action_text} occur."
    )


Case 1 (approved in-scope success): The agent attempted stage_interaction_review(args={"drug_pair": ["Warfarin", "Ibuprofen"], "evidence_id": "ddi:1", "request_id": "request-001", "target": "interaction_review_queue"}); the gate returned allowed; the tool result was {"request_id": "request-001", "status": "staged_in_simulation"}; state review_queue changed from [] to [{"drug_pair": ["Warfarin", "Ibuprofen"], "evidence_id": "ddi:1", "purpose": "clinician review only", "request_id": "request-001"}]; therefore the action did occur.
Case 2 (no approval): The agent attempted stage_interaction_review(args={"drug_pair": ["Warfarin", "Ibuprofen"], "evidence_id": "ddi:1", "request_id": "request-002", "target": "interaction_review_queue"}); the gate returned approval_required; the tool result was not produced (gate blocked before tool invocation); state review_queue changed from [] to []; therefore the action did not occur.
Case 3 (out-of-scope target): The agent attempted stage_interaction_revi